In [2]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV

df = pd.read_csv("academic_meals_elementary_district.csv")

OUTCOMES = ["ELA_Proficiency", "Math_Proficiency", "Science_Proficiency"]

DROP_ALWAYS = set(["District", "missing_frac", "missing_count"] + OUTCOMES)

predictor_cols = [c for c in df.columns if c not in DROP_ALWAYS]
predictor_cols = [c for c in predictor_cols if pd.api.types.is_numeric_dtype(df[c])]

print("Predictors:", len(predictor_cols))

Predictors: 257


In [5]:
#5fold CV
from sklearn.model_selection import KFold

def run_lasso_for_outcome(y_col, predictor_cols, cv_splits=5):
    tmp = df[[y_col] + predictor_cols].copy()
    tmp = tmp.apply(pd.to_numeric, errors="coerce")

    X = tmp[predictor_cols]
    y = tmp[y_col].values

    pipe = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("lasso", LassoCV(
            cv=KFold(n_splits=cv_splits, shuffle=True, random_state=0),
            random_state=0,
            max_iter=200000,
            n_alphas=200
        ))
    ])

    pipe.fit(X, y)
    lasso = pipe.named_steps["lasso"]

    coefs = pd.Series(lasso.coef_, index=predictor_cols)
    selected = coefs[coefs.abs() > 1e-8].sort_values(key=np.abs, ascending=False)

    return {
        "outcome": y_col,
        "n": len(tmp),
        "alpha": lasso.alpha_,
        "n_selected": selected.shape[0],
        "selected_coefs": selected
    }

In [8]:
results = []
for y in OUTCOMES:
    res = run_lasso_for_outcome(y, predictor_cols, cv_splits=5)
    results.append(res)

    print("="*80)
    print(f"Outcome: {res['outcome']} | N={res['n']} | alpha={res['alpha']:.6f} | selected={res['n_selected']}")
    print("Top selected predictors (by |coef|):")
    print(res["selected_coefs"].head(25))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:1663: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(


Outcome: ELA_Proficiency | N=54 | alpha=1.892651 | selected=7
Top selected predictors (by |coef|):
ELA_Proficiency_LowIncome                            11.364835
student_low_income_percent                           -4.593471
ELA_Proficiency_CWD                                   1.071415
Total Fruit Servings in cup equivalents              -0.862751
Refined Grains in ounce equivalents per 1000 kcal     0.569160
Math_Proficiency_LowIncome                            0.436984
Biochanin A (mg)                                      0.045209
dtype: float64


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:1663: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(


Outcome: Math_Proficiency | N=54 | alpha=1.222903 | selected=11
Top selected predictors (by |coef|):
Math_Proficiency_LowIncome                                               11.460118
student_low_income_percent                                               -5.068314
Total Fruit Servings in cup equivalents                                  -1.401689
HEI 2015 Refined Grains (0-10)                                           -1.123701
Math_Proficiency_CWD                                                      1.044006
ELA_Proficiency_CWD                                                       0.781406
Seafood and Plant Protein Servings in ounce equivalents per 1000 kcal     0.307902
MUFA 22:1 (erucic acid) (g)                                              -0.277402
Biochanin A (mg)                                                          0.251943
Pinoresinol (mcg)                                                         0.229252
Total Fruit Servings in cup equivalents per 1000 kcal                

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:1663: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(


Outcome: Science_Proficiency | N=54 | alpha=1.000646 | selected=19
Top selected predictors (by |coef|):
Science_Proficiency_LowIncome                                                        6.019339
student_low_income_percent                                                          -5.313333
Math_Proficiency_LowIncome                                                           3.329630
ROE                                                                                 -2.074032
HEI 2015 Refined Grains (0-10)                                                      -1.386634
ELA_Proficiency_LowIncome                                                            1.324881
MUFA 22:1 (erucic acid) (g)                                                         -1.215238
ELA_Proficiency_CWD                                                                  1.138077
Science_Proficiency_CWD                                                              0.701144
Formononetin (mg)                                 

In [7]:
SES_VARS = [c for c in ["student_low_income_percent","student_disabilities_percent","ROE"] if c in df.columns]
HEI_COMP = [c for c in df.columns if str(c).startswith("HEI 2015") and "Total Score" not in str(c)]
predictor_cols_small = [c for c in SES_VARS + HEI_COMP if c in df.columns]

for y in OUTCOMES:
    res = run_lasso_for_outcome(y, predictor_cols_small, cv_splits=5)
    print("\n" + "="*80)
    print(f"[SES+HEI components] Outcome: {y} | alpha={res['alpha']:.6f} | selected={res['n_selected']}")
    print(res["selected_coefs"])


[SES+HEI components] Outcome: ELA_Proficiency | alpha=2.804955 | selected=2
student_low_income_percent       -12.651932
HEI 2015 Refined Grains (0-10)    -1.483672
dtype: float64

[SES+HEI components] Outcome: Math_Proficiency | alpha=1.000171 | selected=8
student_low_income_percent           -13.800440
HEI 2015 Refined Grains (0-10)        -3.500477
HEI 2015 Saturated Fats (0-10)         3.405251
HEI 2015 Whole Grains (0-10)          -2.977331
HEI 2015 Total Fruits (0-5)           -1.565308
student_disabilities_percent          -1.015808
HEI 2015 Total Protein Foods (0-5)    -0.712879
HEI 2015 Greens and Beans (0-5)       -0.407452
dtype: float64

[SES+HEI components] Outcome: Science_Proficiency | alpha=0.798907 | selected=9
student_low_income_percent           -11.839103
ROE                                   -5.176141
HEI 2015 Whole Grains (0-10)          -4.338720
HEI 2015 Refined Grains (0-10)        -2.892887
HEI 2015 Saturated Fats (0-10)         2.296930
student_disabilities_p

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:1663: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:1663: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib